# Practical P11: Unit 1 Consolidation: AI Data Prep Pipeline
**Learning Outcome**: Build an end-to-end data prep pipeline linking pandas, logging, json, os/pathlib, and custom exception handling.

## Overview of the Pipeline
You are requested to write a complete data preparation script representing a pipeline. The pipeline should:
1. **Load** the messy CSV reviews file using pandas.
2. **Clean** the text columns (handle null values, strip whitespaces, convert to lowercase).
3. **Extract** the reviews and **chunk** them into sections containing no more than 30 words each (representing LLM prompts).
4. **Format** the results into a JSON array.
5. **Save** the output to `data/final_processed_data.json`.
6. Include structured logging alerts throughout execution, and wrap file/data reading operations inside a try-except block.


In [ ]:
import os
import json
import logging
from pathlib import Path
import pandas as pd

# Set up logger for pipeline monitoring
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger('Pipeline')


## Pipeline Blueprint Implementation
Let's write the complete code containing helper functions for each step.


In [ ]:
def load_dataset(file_path):
    logger.info(f'Loading raw dataset from {file_path}')
    try:
        df = pd.read_csv(file_path)
        logger.info(f'Dataset loaded successfully with {len(df)} rows.')
        return df
    except FileNotFoundError as e:
        logger.error(f'Failed to locate dataset: {e}')
        raise

def preprocess_data(df):
    logger.info('Starting preprocessing operations on DataFrame.')
    df['review_text'] = df['review_text'].fillna('')
    df['cleaned_text'] = df['review_text'].str.strip().str.lower()
    # Drop rows that are completely blank
    df = df[df['cleaned_text'] != '']
    logger.info(f'Preprocessing complete. Valid rows remaining: {len(df)}.')
    return df

def chunk_review_text(text, word_limit=30):
    words = text.split()
    chunks = []
    for i in range(0, len(words), word_limit):
        chunks.append(' '.join(words[i:i+word_limit]))
    return chunks

def execute_pipeline(input_csv, output_json):
    logger.info('--- Beginning Pipeline Execution ---')
    try:
        raw_df = load_dataset(input_csv)
        clean_df = preprocess_data(raw_df)
        
        processed_records = []
        for idx, row in clean_df.iterrows():
            review_chunks = chunk_review_text(row['cleaned_text'], word_limit=15)
            for chunk_idx, chunk in enumerate(review_chunks):
                processed_records.append({
                    'review_id': row['review_id'],
                    'customer': row['customer_name'],
                    'chunk_index': chunk_idx,
                    'content': chunk
                })
        
        # Save array to json
        logger.info(f'Saving {len(processed_records)} formatted prompt chunks to {output_json}')
        with open(output_json, 'w') as f:
            json.dump(processed_records, f, indent=2)
            
        logger.info('--- Pipeline Successfully Concluded ---')
    except Exception as e:
        logger.critical(f'Pipeline execution crashed: {e}')

# Run pipeline
execute_pipeline('data/customer_reviews.csv', 'data/final_processed_data.json')


## Hands-On Final Project Challenge
**Challenge**: You are tasked with adding a verification check validation script. Swap scripts with your peer (or test your own script) by writing a short block to load `data/final_processed_data.json` and verifying that:
1. The output file exists.
2. It is a valid JSON list.
3. Each entry in the JSON contains the keys `'review_id'`, `'customer'`, and `'content'`.
Print the first 3 entries of the verified pipeline output.


In [ ]:
# TODO: Complete verification routine
output_path = Path('data/final_processed_data.json')
assert output_path.exists(), 'Output file does not exist!'

with open(output_path, 'r') as f:
    data = json.load(f)

assert isinstance(data, list), 'Output is not a JSON list!'
print(f'Verified successfully. Found {len(data)} items.')

for item in data[:3]:
    assert 'review_id' in item
    assert 'customer' in item
    assert 'content' in item
    print(f"ID {item['review_id']} - {item['customer']}: '{item['content']}'")
